# Exploratory Data Analysis

This notebook examines the cleaned VNAT monthly segment dataset. Segments are used for market composition and recovery interpretation, not as primary forecasting targets.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, MAIN_TARGET, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style
set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

## Total Arrivals Trend

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.index, df[MAIN_TARGET], color=SERIES_COLORS[MAIN_TARGET])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening", "Tourism recovery period"])
ax.set_title("Long-Run Trend in International Tourist Arrivals")
ax.set_ylabel("Monthly arrivals")
save_figure(fig, FIGURES / "01_total_arrivals_timeseries.png")
plt.show()

Observation -> total arrivals show expansion, collapse, and recovery. Statistical implication -> the mean process is not stable over the full sample. Tourism implication -> forecasts require explicit treatment of shock-sensitive recovery dynamics.

## Segment Arrivals Trend

In [ ]:
fig, ax = plt.subplots()
for col in SEGMENT_COLUMNS:
    ax.plot(df.index, df[col], color=SERIES_COLORS[col], label=SERIES_LABELS[col], alpha=0.9)
ax.set_title("Regional Arrival Trends by Source Market")
ax.set_ylabel("Monthly arrivals")
ax.legend(ncol=3, fontsize=8)
save_figure(fig, FIGURES / "02_segment_arrivals.png")
plt.show()

Observation -> Asia is the dominant source-market group, while other regions contribute smaller volumes. Statistical implication -> aggregate arrivals are compositionally concentrated. Tourism implication -> source-market strategy should account for Asian-market dependence.

## Segment Share Over Time

In [ ]:
shares = df[SEGMENT_COLUMNS].div(df[MAIN_TARGET], axis=0) * 100
fig, ax = plt.subplots()
bottom = np.zeros(len(shares))
for col in SEGMENT_COLUMNS:
    vals = shares[col].to_numpy()
    ax.fill_between(shares.index, bottom, bottom + vals, color=SERIES_COLORS[col], alpha=0.75, label=SERIES_LABELS[col])
    bottom += vals
ax.set_ylim(0, 100)
ax.set_title("Source-Market Composition of International Arrivals")
ax.set_ylabel("Share of total arrivals (%)")
ax.legend(ncol=3, fontsize=8)
save_figure(fig, FIGURES / "03_segment_share_over_time.png")
plt.show()

Observation -> market shares shift around the pandemic and reopening period. Statistical implication -> composition changes can affect aggregate forecast stability. Tourism implication -> recovery policy should distinguish volume recovery from source-market reallocation.

## Yearly Pattern

In [ ]:
annual = df[MAIN_TARGET].resample("YS").sum()
fig, ax = plt.subplots()
ax.plot(annual.index.year, annual.values, color=SERIES_COLORS[MAIN_TARGET], marker="o")
ax.set_title("Annual Pattern of International Tourist Arrivals")
ax.set_ylabel("Annual arrivals")
save_figure(fig, FIGURES / "05_yearly_trend.png")
plt.show()

Observation -> annual arrivals collapse during the pandemic and rise during reopening. Statistical implication -> the annual trend contains a structural break. Tourism implication -> pre-pandemic trend extrapolation is not sufficient for planning.

## Monthly Seasonality

In [ ]:
monthly = df.assign(month=df.index.month).groupby("month")[MAIN_TARGET]
median = monthly.median()
q1 = monthly.quantile(0.25)
q3 = monthly.quantile(0.75)
fig, ax = plt.subplots()
ax.plot(median.index, median.values, color=SERIES_COLORS[MAIN_TARGET])
ax.fill_between(median.index, q1.values, q3.values, color=SERIES_COLORS[MAIN_TARGET], alpha=0.18)
ax.set_xticks(range(1, 13))
ax.set_title("Seasonal Dynamics of International Tourist Arrivals")
ax.set_ylabel("Median monthly arrivals")
save_figure(fig, FIGURES / "04_monthly_seasonality.png")
plt.show()

Observation -> arrivals exhibit recurring within-year variation. Statistical implication -> seasonal terms are necessary in forecasting models. Tourism implication -> staffing, aviation capacity, and destination services should align with predictable seasonal demand.

## Pre-COVID, COVID, and Post-Reopening Comparison

In [ ]:
period = pd.Series(index=df.index, dtype="object")
period.loc[:"2020-02-01"] = "Pre-COVID"
period.loc["2020-03-01":"2022-02-01"] = "COVID"
period.loc["2022-03-01":] = "Post-reopening"
period_month = df.assign(period=period, month=df.index.month).groupby(["period", "month"])[MAIN_TARGET].mean().unstack(0)
fig, ax = plt.subplots()
colors = {"Pre-COVID":"#2f3437", "COVID":"#9a9a9a", "Post-reopening":"#4e79a7"}
for col in ["Pre-COVID", "COVID", "Post-reopening"]:
    if col in period_month:
        ax.plot(period_month.index, period_month[col], label=col, color=colors[col])
ax.set_xticks(range(1, 13))
ax.set_title("Seasonality Across Pandemic Regimes")
ax.set_ylabel("Average arrivals")
ax.legend()
save_figure(fig, FIGURES / "16_regime_seasonality_comparison.png")
plt.show()

Observation -> the seasonal profile changes sharply across regimes. Statistical implication -> the pandemic is a structural break, not ordinary noise. Tourism implication -> recovery planning should not assume immediate restoration of pre-COVID seasonality.

## Rolling Volatility Proxy

In [ ]:
growth = df[MAIN_TARGET].pct_change() * 100
vol = growth.rolling(12).std()
fig, ax = plt.subplots()
ax.plot(vol.index, vol, color=SERIES_COLORS[MAIN_TARGET])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening"])
ax.set_title("Rolling Volatility of Monthly Arrival Growth")
ax.set_ylabel("12-month rolling std. dev. (%)")
save_figure(fig, FIGURES / "17_rolling_volatility.png")
plt.show()

Observation -> volatility rises around the shock and recovery periods. Statistical implication -> residual variance is time-varying. Tourism implication -> forecast intervals are as important as point forecasts for capacity decisions.

## Growth-Rate Analysis

In [ ]:
yoy = df[MAIN_TARGET].pct_change(12) * 100
fig, ax = plt.subplots()
ax.axhline(0, color="#777777", linewidth=0.8)
ax.plot(yoy.index, yoy, color=SERIES_COLORS[MAIN_TARGET])
annotate_events(ax, ["COVID-19 outbreak", "Vietnam border reopening"])
ax.set_title("Year-on-Year Growth and Recovery Momentum")
ax.set_ylabel("Year-on-year growth (%)")
save_figure(fig, FIGURES / "06_growth_rate_analysis.png")
plt.show()

Observation -> year-on-year growth is extreme during reopening. Statistical implication -> percentage growth is unstable when the denominator is depressed. Tourism implication -> rebound growth should not be interpreted as normal long-run expansion.

## Segment Correlation

In [ ]:
corr = df[SEGMENT_COLUMNS].pct_change().corr()
fig, ax = plt.subplots(figsize=(6.5, 5.3))
im = ax.imshow(corr, cmap="Greys", vmin=-1, vmax=1)
labels = [SERIES_LABELS[c] for c in SEGMENT_COLUMNS]
ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlation of Segment Arrival Growth")
save_figure(fig, FIGURES / "08_correlation_heatmap.png")
plt.show()

Observation -> segment growth correlations are positive but heterogeneous. Statistical implication -> common shocks coexist with source-market-specific dynamics. Tourism implication -> diversification across source markets can reduce exposure to region-specific volatility.